Option 1: Retrain their entire brain every time

This is full fine-tuning.

Option 2: Keep the brain mostly unchanged and attach a small notebook

This is LoRA.

Instead of changing the whole brain:

"Whenever you see something relevant to sentiment analysis, also consult this small notebook."



Full Fine-Tuning

BERT

 ↓

Every layer updated

 ↓

66M trainable params

LoRA

BERT

 ↓

Frozen

 ↓

Tiny adapters inserted

 ↓

Only adapters trained


Full Fine-Tuning vs LoRA

| Aspect                  | Full Fine-Tuning | LoRA        |
| ----------------------- | ---------------- | ----------- |
| Parameters trained      | 66M              | ~300K       |
| Training speed          | Slower           | Faster      |
| Storage                 | Large            | Tiny        |
| Cost                    | High             | Low         |
| Risk of forgetting      | Higher           | Lower       |
| One model per task      | Yes              | No          |
| Industry usage for LLMs | Rare             | Very common |


What You Should Remember

Only remember these 4 points:

1.

LoRA = Low-Rank Adaptation

2.

Instead of training the whole model:

Train tiny adapters
3.

The base model stays frozen.

4.

LoRA is the standard way to fine-tune modern LLMs because it's:

cheaper
faster
smaller
easier to deploy

If someone says:

"I fine-tuned LLaMA 3 on my custom dataset using a single GPU"

there is a very high chance they used LoRA/PEFT, not full fine-tuning.

In [ ]:
import sys
!{sys.executable} -m pip install peft

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

# Inspect layer names to know which to target
for name, _ in base_model.named_modules():
    if 'attention' in name.lower():
        print(name)


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,        # sequence classification
    r=8,                               # rank — smaller = fewer params, less expressive
    lora_alpha=16,                     # scaling factor (usually 2×r)
    lora_dropout=0.1,                  # dropout on LoRA layers
    target_modules=['q_lin','v_lin'],  # which attention matrices to adapt
    bias='none',                       # don't train bias terms
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

With LoRA:

Input Sentence

      ↓

DistilBERT

      ↓

LoRA Adapters

      ↓
Classifier

      ↓

Prediction

The original DistilBERT stays frozen.

Only tiny LoRA adapters learn.

What You Should Remember

Only remember these five points:

1.

get_peft_model()

→ inserts LoRA adapters into the pretrained model.

2.

r

→ size/capacity of the adapters.

3.

target_modules=['q_lin','v_lin']

→ adapt Query and Value attention matrices because they matter most.

4.

The original DistilBERT weights stay frozen.

5.

You train roughly:

0.5% of parameters

instead of:

100%

and still get nearly the same performance.

That single idea is why LoRA became the standard method for fine-tuning modern LLMs like Llama, Mistral, Gemma, Qwen, and Falcon.

In [ ]:
from datasets import load_dataset
from transformers import DataCollatorWithPadding

dataset   = load_dataset('glue', 'sst2')

def tokenize(batch):
    return tokenizer(batch['sentence'], truncation=True, max_length=128)

tokenized = dataset.map(tokenize, batched=True, remove_columns=['sentence','idx'])
tokenized = tokenized.rename_column('label','labels')
tokenized.set_format('torch')

train_data = tokenized['train'].select(range(4000))
val_data   = tokenized['validation'].select(range(500))

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate, numpy as np

accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1':       f1_metric.compute(predictions=preds, references=labels, average='binary')['f1'],
    }

training_args = TrainingArguments(
    output_dir='./distilbert-lora',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-4,          # LoRA can use higher lr than full fine-tuning
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print("\n=== LoRA Results ===")
print(results)

print("\n=== Comparison ===")
print(f"Full fine-tune (Day 3): ~91% accuracy, ~66M trainable params")
print(f"LoRA (today):           {results['eval_accuracy']*100:.1f}% accuracy, ~300K trainable params")

The only new thing is:

Instead of training all DistilBERT parameters, Trainer is now training only the tiny LoRA adapters.


What You Should Remember

Only remember these 4 things:

1.

The training pipeline is almost identical to Day 3.

2.

The base DistilBERT model stays frozen.

3.

Only LoRA adapters are trained.

4.

Because adapters are small, we can use a much higher learning rate:

Full Fine-Tuning → 2e-5

LoRA → 3e-4

while achieving nearly the same accuracy.

This is why LoRA became the standard way to fine-tune modern LLMs and transformer models.

In [ ]:
# Option 1: save only the adapter weights (~2MB)
# The base model is not saved — loaded separately at inference
model.save_pretrained('./lora-adapter-only')

# This saves only the tiny adapter files:
# adapter_config.json + adapter_model.safetensors

In [ ]:
# Option 2: merge adapters into base model then save
# Result: a normal model file with LoRA baked in — no peft needed at inference
from peft import PeftModel

# Load base + adapter and merge
base   = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
merged = PeftModel.from_pretrained(base, './lora-adapter-only')
merged = merged.merge_and_unload()     # merges A×B into W, removes LoRA structure

merged.save_pretrained('./distilbert-lora-merged')
tokenizer.save_pretrained('./distilbert-lora-merged')
print("Merged model saved")


In [ ]:
from transformers import pipeline

clf = pipeline(
    'text-classification',
    model='./distilbert-lora-merged',
    tokenizer='./distilbert-lora-merged',
    device=-1
)

tests = [
    "An absolute masterpiece from start to finish.",
    "Painfully dull and a complete waste of potential.",
    "Not the worst thing I've seen but close.",
    "Genuinely surprised by how good this was.",
]

for res, text in zip(clf(tests), tests):
    print(f"[{res['label']:8s} {res['score']:.1%}]  {text}")
Check file sizes — the key LoRA advantage
import os

def folder_size(path):
    total = sum(os.path.getsize(os.path.join(p,f))
                for p,_,files in os.walk(path) for f in files)
    return total / 1e6  # MB

print(f"Adapter only:    {folder_size('./lora-adapter-only'):.1f} MB")
print(f"Merged model:    {folder_size('./distilbert-lora-merged'):.1f} MB")

Key Takeaway

LoRA's biggest practical advantage is not just faster training—it is modularity.

You can:

Keep one frozen base model.

Train tiny adapters for different tasks.

Swap adapters whenever needed.

Merge them only when deploying a standalone production model.

This is exactly how many teams fine-tune and deploy modern LLMs such as Llama, Mistral, and other large transformer models.